# M11.13 · Sürekli görevler: ortalama ödül ve diferansiyel kontrol

Doğal bitişi olmayan görevlerde adım başı ödülü optimize et: durağan dağılım, gain, diferansiyel değer ve Poisson denklemi. Periyodik halka/Abel limiti, yarı-gradyan SARSA, 10 sunuculu erişim kontrolü, indirimli amacın kapsamı ve n-adım başlangıç izi. Saf Python geçiş/öğrenme → NumPy tam model → PyTorch stop-gradient → bağımsız ajanları vektörleştirme; altı kaynak sorusu çözümlü.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-13-average-reward.html](https://m-gonul.github.io/pytorch-lab/m11-13-average-reward.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_13_average_reward.py
CHECKS = {
 "b": "from labcheck import close, need, ensure\nimport random\nimport numpy as np\n\n\ndef check(ns, R_):\n    def t():\n        step = need(ns, \"access_step\")\n        close(step(0, 3, 1, random.Random(0), p=1)[:1], [10], what=\"dolu havuzda bütün eski işler biter\")\n        close(step(0, 3, 1, random.Random(0), p=1)[2], 0, what=\"boş yoksa ödül yok\")\n        close(step(10, 2, 1, random.Random(0), p=1, release_new=True)[0], 10, what=\"yeni iş de bitebilir\")\n        close(step(10, 2, 1, random.Random(0), p=1, release_new=False)[0], 9, what=\"yeni iş hariç zamanlama\")\n        close(step(4, 1, 1, random.Random(0), p=0)[::2], [3, 2], what=\"bırakma yok; tek kabul\")\n        out = need(ns, \"differential_sarsa\")(steps=10000, alpha=.02, beta=.005, seed=3)\n        Q = np.array(out[\"Q\"])\n        close(Q.shape, [11, 4, 2], what=\"tablo boyutu\")\n        close(np.sum(out[\"visits\"]), 10000, what=\"her gerçek adımda bir ziyaret\")\n        close(Q[0, :, 1], [0] * 4, what=\"geçersiz kabul güncellenmez\")\n        close(out[\"rho\"], (.005 / .02) * Q.sum(), atol=1e-9, what=\"aynı δ ile gain/ağırlık toplamı değişmezi\")\n        ensure(0 <= out[\"reward_mean\"] <= 8 and np.isfinite(Q).all(), \"gerçek ödül ve sonlu ağırlıklar\")\n        R_.table(\"10000 gerçek geçiş\", [\"ortalama ödül\", \"son rho\", \"ziyaret\"], [[out[\"reward_mean\"], out[\"rho\"], int(np.sum(out[\"visits\"]))]])\n    R_.test(\"Sunucu zamanlaması ve diferansiyel güncelleme\", t)\n",
 "c": "from labcheck import close, need, ensure\nimport numpy as np\n\n\ndef check(ns, R_):\n    def t():\n        P = np.array([[0., 1., 0.], [0., 0., 1.], [1., 0., 0.]])\n        rho, h = need(ns, \"average_evaluate\")(P, [0., 0., 1.])\n        close(rho, 1/3, what=\"üç halkada gain\")\n        close(h, [0, 1/3, 2/3], what=\"hA=0 referanslı Poisson çözümü\")\n        for include in (True, False):\n            D = need(ns, \"access_kernel\")(.06, include)\n            close(D.sum(axis=-1), np.ones((11, 2)), what=\"sıfır sunucuda satırı iki kez toplama hatası yok\")\n            close(D[0, 0], D[0, 1], what=\"zorunlu red geçişi\")\n        results = []\n        for include, eps, expected in [(True, 0., 2.747641950572482), (True, .1, 2.6468943683103894), (False, 0., 2.6731922050681827)]:\n            out = need(ns, \"optimal_access\")(.06, eps, include)\n            close(out[\"rho\"], expected, atol=1e-10, what=\"bağımsız44durum modelinden gain\")\n            close(out[\"r\"] + out[\"P\"] @ out[\"h\"] - out[\"h\"], np.full(11, out[\"rho\"]), atol=1e-10, what=\"Poisson kalanı\")\n            ensure(np.all(out[\"accept\"][0] == 0), \"boş0iken red\")\n            results.append([include, eps, out[\"rho\"]])\n        R_.table(\"Model oracle'ı: kaynakta eksik zamanlama açık\", [\"yeni iş de bitebilir\", \"ε\", \"gain\"], results)\n    R_.test(\"Poisson normalizasyonu ve bağımsız access-control referansı\", t)\n",
 "d": "from labcheck import close, need, ensure\nimport torch\n\n\ndef check(ns, R_):\n    def t():\n        Q = torch.zeros(44, 2, dtype=torch.float64, requires_grad=True)\n        Q.data[1, 0] = 3.0\n        updated, rho = need(ns, \"differential_torch_step\")(Q, .5, 0, 1, 2., 1, 0, .1, .02)\n        # target2-.5+3=4.5; only the current prediction receives a gradient.\n        close(updated[0, 1].detach().numpy(), .45, what=\"doğru current-only adım\")\n        close(updated[1, 0].detach().numpy(), 3., what=\"hedef parametresi değişmedi\")\n        close(rho, .59, what=\"aynı δ ile gain adımı\")\n        ensure(updated.is_leaf and updated.requires_grad, \"yeni adım için temiz yaprak\")\n        out = need(ns, \"torch_sarsa\")(steps=3000, alpha=.02, beta=.005, seed=3)\n        close(sum(out[\"visits\"]), 3000, what=\"fiziksel geçişler\")\n        close(out[\"rho\"], .25 * float(out[\"Q\"].sum()), atol=1e-9, what=\"gain / toplam ağırlık değişmezi\")\n        close(out[\"Q\"][:4, 1].numpy(), [0] * 4, what=\"boş0geçersiz kabul\")\n        ensure(bool(torch.isfinite(out[\"Q\"]).all()) and 0<=out[\"reward_mean\"]<=8, \"sonlu değerler\")\n        R_.table(\"3000 geçiş / autograd\", [\"gerçek ortalama\", \"rho\"], [[out[\"reward_mean\"], out[\"rho\"]]])\n    R_.test(\"Detach desteği ve gerçek diferansiyel kontrol\", t)\n",
 "e": "from labcheck import close, need, ensure, max_loops\nimport torch\n\n\ndef check(ns, R_):\n    R_.test(\"ortam döngüsü yok\", lambda: max_loops(ns[\"__source__\"], 1, func=\"batched_access\"))\n    def t():\n        out = need(ns, \"batched_access\")(n_envs=16, steps=3000, alpha=.02, beta=.005, seed=3)\n        ensure(tuple(out[\"Q\"].shape)==(16,44,2), \"ayrı Q tabloları\")\n        close(out[\"visits\"].sum(dim=1).numpy(), [3000]*16, what=\"ortam başına3000gerçek geçiş\")\n        close(out[\"rho\"].numpy(), (.25*out[\"Q\"].sum(dim=(1,2))).numpy(), atol=1e-9, what=\"her ortamda ayrı gain değişmezi\")\n        close(out[\"Q\"][:, :4, 1].numpy(), torch.zeros(16,4).numpy(), what=\"geçersiz eylemler\")\n        ensure(bool(torch.isfinite(out[\"Q\"]).all()) and bool((out[\"reward_mean\"]>=0).all()) and bool((out[\"reward_mean\"]<=8).all()), \"sonlu sonuçlar\")\n        ensure(float(out[\"reward_mean\"].std())>0, \"bağımsız ortamlar farklı sonuç üretiyor\")\n        R_.table(\"16ajan ×3000geçiş\", [\"ortalama gerçek ödül\", \"koşular arası std\"], [[float(out[\"reward_mean\"].mean()),float(out[\"reward_mean\"].std())]])\n    R_.test(\"Bağımsız vektörleştirilmiş kontrol\", t)\n",
 "x1": "from labcheck import close, need, ensure\nimport numpy as np\n\n\ndef check(ns, R_):\n    def t():\n        gain = need(ns, \"gain_step\")\n        close(gain(10., -18., 2, .1, 0., True), [1., .1], what=\"ilk normalize hedef; n ölçeği\")\n        close(gain(-100., 202., 2, .1, 0., True)[0], 1., what=\"aynı hedefin farklı başlangıç katsayısı\")\n        close(gain(10., -18., 2, .1, 0., False)[0], 8.2, what=\"kutudaki düz beta-delta sürümü\")\n        rho, trace = -17., 0.\n        for t in range(1, 21):\n            rho, trace = gain(rho, 8*(2-rho), 8, .1, trace, True)\n            close(rho, 2., what=\"sabit ortalama hedef\")\n            close(trace, 1-.9**t, what=\"iz kütlesi\")\n        out = need(ns, \"differential_n_sarsa\")(steps=2000,n=8,seed=2)\n        close(out[\"updates\"], 1993, what=\"halka tamponu ısınma gecikmesi\")\n        close(np.sum(out[\"visits\"]), 2000, what=\"gerçek fiziksel adımlar\")\n        ensure(np.isfinite(np.asarray(out[\"Q\"])).all() and np.isfinite(out[\"rho\"]), \"sonlu sonuçlar\")\n        R_.table(\"n ölçeği, ilk hata−18\", [\"naif iz düzeltmesi\", \"ortalama-hedef düzeltmesi\"], [[10+1*(-18),gain(10.,-18.,2,.1,0.,True)[0]]])\n    R_.test(\"Gain başlangıç katsayısı ve n-adım tamponu\", t)\n",
 "x2": "from labcheck import close, need, ensure\nimport numpy as np\n\n\ndef check(ns, R_):\n    def t():\n        out = need(ns, \"differential_q_demo\")(steps=50000,seed=0,eta=.1)\n        Q = np.asarray(out[\"Q\"])\n        close(np.sum(out[\"visits\"]),50000,what=\"bütün gerçek adımlar\")\n        ensure(np.min(out[\"visits\"])>1000,\"bütün durum-eylem çiftleri kapsanıyor\")\n        close(out[\"rho\"], .1*Q.sum(),atol=1e-9,what=\"aynı hata/adım oranı değişmezi\")\n        ensure(Q[0,1]>Q[0,0],\"en iyi gain politikasında S0→S1\")\n        close(out[\"rho\"],8/3,atol=.15,what=\"modelin tam gain değeri, sonlu örnek payıyla\")\n        ensure(abs(out[\"behavior_reward\"]-out[\"rho\"])>.1,\"davranış ödülü ve hedef gain ayrılmalı\")\n        R_.table(\"50000 geçiş; farklı hedef/davranış\",[\"hedef gain\", \"gerçek davranış ortalaması\", \"tam optimum\"],[[out[\"rho\"],out[\"behavior_reward\"],8/3]])\n    R_.test(\"Diferansiyel Q: kapsam ve doğru hedef\",t)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_13_average_reward as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir sunucu kuyruğunun doğal sonu yoktur. Yalnız ilk müşterinin veya yakındaki ödülün değerini büyütmek yerine **adım başına ortalama ödülü** artırabiliriz. Bu amaçta γ yoktur; her gelecek adım aynı ağırlıktadır. Değer fonksiyonu, geleceğin toplam ödülünü değil, uzun dönem ortalamasına göre önden veya geriden gitmeyi ölçer.

Önce üç durumlu halkayı elle çözün. Sonra saf Python ile diferansiyel SARSA, NumPy ile bağımsız model hesabı, PyTorch ile hedefin türevini durdurma ve birçok bağımsız ajanı vektörleştirme geliyor. Model yalnız doğrulama aracıdır; öğrenen sunucunun boşalma olasılığını bilmez. Episodik Mountain Car kontrolü [M11.6](m11-6-dqn.html#sutton-kontrol-hedef) içindedir.

## 1 · Elle

**Üç halka denklemini yazın.** A→B ve B→C ödül0, C→A ödül1. Bir tur üç adım: ρ=1/3. Diferansiyel Bellman denklemi her durumda `h=reward−rho+h_next`; çözüm sabite kadar belirlenir. `µᵀh=0` seçince h=(−1/3,0,+1/3).

**Limitleri ayırın.** +1,0 dönüşümlü ödülde ortalama1/2 ama merkezli toplam salınır. Abel hesabında önce sabit γ<1 ile sonsuz geometrik toplamı, sonra γ↑1 limitini alın.

**İzde n katsayısını kaçırmayın.** n2, başlangıç gain10, toplam ödül2 ve Q farkı0 için δ=−18. İlk iz o=β; ham δ kullanan naif güncelleme−8, ortalama hedefini kullanan doğru güncelleme1 verir. Aşağıdaki alanları bu denklemlerle doldurun.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**A→B→C→A; yalnız A varışı +1**

- `rho3`: Ortalama ödül ρ
- `ha`: µᵀh=0 için h(A)
- `hc`: µᵀh=0 için h(C)

**A ilk1, B ilk0; ödüller dönüşümlü**

- `rho2`: Ortalama ödül
- `abel`: Abel limiti h(A)

**n2, β0.1, Rbar0=10, ödül toplamı2, Q farkı0**

- `delta`: Ham δ
- `bad`: İlk naif iz sonrası Rbar
- `good`: İlk doğru iz sonrası Rbar

In [ ]:
cevaplar = dict(
    rho3=None,
    ha=None,
    hc=None,
    rho2=None,
    abel=None,
    delta=None,
    bad=None,
    good=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"rho3":[0.3333333333333333,0],"ha":[-0.3333333333333333,0],"hc":[0.3333333333333333,0],"rho2":[0.5,0],"abel":[0.25,0],"delta":[-18,0],"bad":[-8,0],"good":[1,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

**Kuyruk hiç bitmiyor.**10 sunucu, dört eşit öncelik, kabul ödülleri1/2/4/8. Her adımda bir müşteri kabul veya reddedilir. Boş sunucu yoksa kabul eylemi de zorunlu reddir. `access_step` yeni boş sayısını, önceliği ve ödülü döndürsün.

Kaynak ayrı ayrı her meşgul sunucu için bırakma olasılığı.06 verir. Burada bırakmaları **bağımsız Bernoulli** seçiyoruz. `release_new=True` durumunda yeni kabul edilen iş de aynı adımın sonunda bitebilir; `False` yalnız önceden meşgul sunucuları bırakır. Bu zamanlama, kaynakta açıkça belirlenmeyen bir model tercihidir.

`differential_sarsa` gerçek geçişlerden öğrenir; modeli kullanmaz. Tablo `Q[free][priority][action]`, başlangıç0; ε-greedy eşitlikleri rastgele. Terminal, reset veya γ yok. Sonraki eylemi **ağırlıkları değiştirmeden önce** seçin. Tek hata `delta=reward-rho+Qnext-Qcurrent` hem `rho += beta*delta` hem `Q += alpha*delta` için kullanılır. Çıktı `Q`, `rho`, gerçek ödül ortalaması `reward_mean` ve `visits` sözlüğü olsun. Kaynak: Sutton–Barto, s.251, Örnek10.2 ve diferansiyel SARSA kutusu.

In [ ]:
%%writefile stage_b.py
def access_step(free, priority, action, rng, p=0.06, release_new=True):
    # TODO: kabul/ödül, meşgul sunucuların bağımsız bırakılması, yeni öncelik
    raise NotImplementedError


def differential_sarsa(steps=20000, alpha=0.01, beta=0.01, epsilon=0.1,
                       seed=0, p=0.06, release_new=True):
    # TODO: sonraki eylemi güncellemeden önce seç; aynı delta ile rho ve Q
    raise NotImplementedError

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

**Modeli yalnız doğrulamak için kullan.** Ajanın bilmediği bırakma olasılığını bu aşamada doğrulama modeli kurmak için biliyoruz. `access_kernel`11×2×11 geçiş matrisi üretsin; yeni iş zamanlaması b aşamasıyla aynı olsun. Her satırın toplamı1 olmalı. Boş sunucu0 durumunda iki eylemin çevre geçişi aynıdır, ama politika yalnız reddi seçer.

Öncelikler her yeni adımda bağımsız ve düzgün olduğundan, politika değerlendirmesini **öncelik gelmeden önceki**11 boş-sunucu durumuna indirebilirsin. `accept[f,priority]` kabul olasılığı olsun; öncelik üzerinde ortalama alarak P ve r kur. `average_evaluate` Poisson sistemini `(I-P)h+rho*1=r`, `h[anchor]=0` ile çözsün. Dönen h, öncelik henüz bilinmeden koşullanan değerdir;44durumlu değer tablosuyla aynı nesne değildir.

`optimal_access` bu modelde politika yinelemesi yapsın. `epsilon=0` tüm deterministik politikaları; pozitif ε her geçerli iki-eylemli durumda en az ε/2 keşif olasılığı olan aileyi optimize eder. İlk satır zorunlu red. Çıktı `rho,h,accept,greedy,P,r`. Sonlu modelde politika değerlendirmesi/iyileştirmesi bir **oracle** sağlar; öğrenenin bu matrisi gördüğü iddia edilmez. Bellman'a göre iyileştirme tam modelde yapılır; yaklaşık q için genel iyileştirme güvencesine karıştırılmaz.

In [ ]:
%%writefile stage_c.py
def average_evaluate(P, r, anchor=0):
    # TODO: (I-P)h + rho*1 = r; h[anchor]=0 kısıtı
    raise NotImplementedError


def access_kernel(p=0.06, release_new=True):
    # TODO: her (boş,eylem) için toplamı1 olan binom geçişi
    raise NotImplementedError


def optimal_access(p=0.06, epsilon=0.0, release_new=True):
    # TODO: 11 boş-sunucu durumunda politika değerlendirme/iyileştirme
    raise NotImplementedError

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

**Yarı gradyanı autograd ile kur.** `Q`44×2 boyutlu, `float64` yaprak tensor olsun: durum indisi `4*free+priority`. Bu, bir-sıcak özellikli doğrusal modelin tabloya denk özel durumudur. `differential_torch_step` yalnız `Q[state,action]` üzerinden yarım kare kaybı türevlesin. Sonraki Q **detach** edilmelidir; ortalama ödül kestirimi de bir optimizasyon parametresi değildir.

Fonksiyon yeni yaprak Q ve yeni rho döndürsün. Aynı TD hatasının sayısal değeri rho güncellemesinde kullanılır. `torch_sarsa` bu adımı kaynak ortamında5000 gerçek geçişte uygulasın; `Q,rho,reward_mean,visits` döndürsün. Yeni eylem güncellemeden önce seçilir. `visits`44elemanlıdır. Yeni kabul edilen işi de bırakma çekilişine kat; p=.06. Bu aşama model-oracle'ına bakmaz.

Hedefi türevlemek yalnız bir API hatası değildir: sonraki durum parametrelerini de değiştirir ve kaynak yarı-gradyan kuralını başka bir algoritmaya çevirir. Kontrol, basit bir geçişin doğru gradyan desteğini ayrıca denetler.

In [ ]:
%%writefile stage_d.py
def differential_torch_step(Q, rho, state, action, reward, next_state, next_action,
                            alpha=0.01, beta=0.01):
    # TODO: hedefi detach et; yarım kare kayıptan yalnız mevcut Q için gradyan
    raise NotImplementedError


def torch_sarsa(steps=5000, alpha=0.01, beta=0.01, epsilon=0.1, seed=0):
    # TODO: sürekli on-policy kontrol, terminal/reset/indirim yok
    raise NotImplementedError

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**Aynı zamanı paylaşan, ayrı öğrenen ajanlar.** `batched_access` B bağımsız ortamın tablolarını `Q[B,44,2]`, gain kestirimlerini `rho[B]` ve ziyaretlerini `visits[B,44]` biçiminde taşısın. Yalnız zaman döngüsü kalsın. Her adımda bütün ortamlarda bırakmaları B×10 Bernoulli matrisiyle çek, gerçekten meşgul sunucularla maskele. Öncelikler düzgün0..3. Yeni işi aynı adımda bitebilir kabul ediyoruz.

Bir mini yığından tek ortak ağ öğrenmiyoruz: her ortam kendi ağırlığını günceller. Bu yüzden ortalama kayıpla B'ye bölünmüş ortak gradyan yok; her satır için `Q[b,s,a] += alpha*delta[b]`. Gain ve sonraki eylemler de bağımsızdır. Büyüklüğü Bolan indeks dizisiyle gather/scatter kullan. Dönen sözlük `Q,rho,reward_mean,visits`; bütün sayılar ortam başına verilir.

Rastgele sayılar vektör halinde farklı sırada tüketildiğinden aynı seed, döngülü b aşamasıyla birebir aynı yolları vaat etmez. Birden çok bağımsız koşu üretme amacı ve güncelleme kuralları aynıdır. Hız iddiası için aynı donanımda ölçüm gerekir.

In [ ]:
%%writefile stage_e.py
def batched_access(n_envs=64, steps=5000, alpha=0.01, beta=0.01, epsilon=0.1, seed=0):
    # TODO: yalnız zaman döngüsü; her ortamın ayrı Q, gain ve ziyareti
    raise NotImplementedError

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · 10.9: gerçek n-adım diferansiyel SARSA ve gain izi

**n-adım ve başlangıç katsayısı.** Kaynak: Sutton–Barto, s.255, (10.14–10.15), algoritma kutusu ve Alıştırma10.9. Ortam b aşamasındakiyle aynıdır; yeni iş bırakma çekilişine dahildir. `differential_n_sarsa` yalnız n+1 uzunluklu durum/eylem/ödül tamponu kullansın. Terminal veya yapay reset yok; T gerçek adımdan sonra en çok `T-n+1` öğrenme güncellemesi yapılır. Sondaki henüz dolmamış hedefler yeni ödül uydurularak tamamlanmaz.

Raw hata `delta=sum(n rewards)-n*rho+Qbootstrap-Qcurrent`. Düz sürüm `rho += beta*delta` kullanır. Başlangıç katsayısını kaldıran sürümde `trace += beta*(1-trace)`, `rho += beta/trace*(delta/n)` kullan. Q güncellemesi **orijinal raw delta** ile kalır. Bu sürümde β, n-ödüllü **ortalama hedefin** adımıdır; kutudaki raw-δ adımına göre etkin katsayı β/n olur. Kutudaki β'yı aynen korumak istersen izde nβ kullan; convex ağırlık yorumu için0<nβ≤1 gerekir.

Neden gerekli: n=2,rho0=10,sumR=2,Qfarkı0 için ilk rawδ=−18. Naif `beta/trace*delta`, ilk adımda−8 verir; doğru normalize hedef1 verir. Bu, verilen hedef dizisinde başlangıç katsayısını kaldırır. Öğrenilen Q ve politika hedefi değiştirdiğinden genel gain kestirimini her anda yansız yapma garantisi değildir. `gain_step` yeni rho ve izi; ana fonksiyon Q,rho,trace,reward_mean,visits,updates döndürsün.

In [ ]:
%%writefile extra_x1.py
def gain_step(rho, delta, n, beta, trace, debias=True):
    # TODO: raw delta içinde n*rho var; ortalama hedefi için delta/n
    raise NotImplementedError


def differential_n_sarsa(steps=5000, n=2, alpha=.01, beta=.01, epsilon=.1,
                         seed=0, debias=True, initial_gain=0.0):
    # TODO: n+1 halka tamponu, ortak güncel rho, sürekli n-adım SARSA
    raise NotImplementedError

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · 10.4: tablosal Differential Q-learning, azalan adım

**Diferansiyel Q-learning.** Alıştırma10.4 pseudocode'unun küçük tablo örneği. S0 durumunda eylem0ödül2verip S0kalır; eylem1ödül0ile S1geçer. S1 durumunda iki eylem de ödül4verir; sonraki durum yarı olasılıkla S0/S1. Davranış her eylemi düzgün seçsin. Hedef `reward-rho+maxQ(next)-Qcurrent`; Q için α, gain için `eta*alpha` aynı hatayı kullanır. Her çiftin α değeri `.2/(1+ziyaret)^.6` olsun.

Gerçek en iyi ortalama ödül8/3; davranış politikası aynı değildir. Rastgele davranışın gerçek ödül ortalamasını gain hedefiyle karıştırma. Çıktı `Q,rho,visits,behavior_reward`. Bütün çiftler sonsuza kadar örneklenirse bu küçük model, iletişim/tek-sabit-farkı çözümü ve uygun azalan adım koşullu tablo teoremine uygundur. Sabit adımlı genel fonksiyon yaklaşımı için güvence çıkarılmaz.

Kaynak çözümünü doğrulamak için ek okuma: Wan–Naik–Sutton (ICML2021), (2–4). Bu çalışma gain güncellemesini yalnız davranışın açgözlü eylemlerine sınırlamaz. Ek kaynak burada bütünüyle içe aktarılmış sayılmaz.

In [ ]:
%%writefile extra_x2.py
def differential_q_demo(steps=50000, seed=0, eta=0.1):
    # TODO: davranış eylemi düzgün, hedef maxQ; gain aynı delta ile güncellenir
    raise NotImplementedError

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

## Makale ödevi: Erişim kontrolünde zamanlama ve n-adım karşılaştırması

[Sutton–Barto, Örnek10.2 ve Alıştırma10.9](http://incompleteideas.net/book/the-book-2nd.html)

Her iki boşalma sırası için n1/2/4/8, en az10 tohum ve her koşu2m gerçek geçişle diferansiyel SARSA karşılaştır. Öğrenen p bilmesin; son politikayı oracle ile ayrı değerlendir.

**Adımlar**

1. α,β ve iz tanımını sonuçlara bakmadan seç; n ölçeğini kaydet.
2. Rbar, gerçek ortalama, son blok ve dondurulmuş politika gain ölç; ziyaretleri sakla.
3. Aynı normalizasyonla Q ve Poisson değerlerini karşılaştır; az görülen durumları işaretle.

**Kabul ölçütleri**

- Tüm fiziksel adımlar sayılmış; eksik koşular gizlenmemiş.
- Tohumlar arası SE ve her modelin kesin gain referansı raporlanmış.
- Sonlu sabit-adım deneyinden genel yakınsama teoremi çıkarılmamış.

## Kaynaklar

- [Sutton & Barto — Reinforcement Learning, 2e, Bölüm10 (s.243–256)](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0
- [Wan, Naik, Sutton (2021) — Learning and Planning in Average-Reward Markov Decision Processes (yalnız tablosal Q doğrulaması)](https://proceedings.mlr.press/v139/wan21a.html) — Lisans doğrulanmadı